In [53]:
#%%time
import os
import re
import gzip
import math
import random
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from ast import literal_eval
import warnings
# 忽略 FutureWarning 类型警告
warnings.simplefilter(action='ignore', category=FutureWarning)
# 忽略特定类型的警告：忽略 scanpy包中含有 ignore 的 UserWarning 类型警告
warnings.filterwarnings("ignore", category=UserWarning, module="scanpy")
# 禁用 pandas 包中的 SettingWithCopyWarning 类型警告  
pd.options.mode.chained_assignment = None  # 或 'raise' 表示引发异常

import inferECC
from inferECC import *

In [54]:
#### 20250902 更新 Fig1 eCDNA hub 信息

In [55]:
def get_all_folders(path):
    folders = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
    return folders
"""
# 指定路径
target_path = library_dir # '/your/target/directory'
# 获取所有文件夹
folders_list = get_all_folders(target_path)
"""

"\n# 指定路径\ntarget_path = library_dir # '/your/target/directory'\n# 获取所有文件夹\nfolders_list = get_all_folders(target_path)\n"

In [105]:
import re
from typing import Iterable, List, Tuple

def _chrom_key(chr_str: str) -> Tuple[int, str]:
    """将 chr 编号转为可排序的键：chr1..chr22, X, Y, M/MT 置前，其它按字典序置后。"""
    _chr_map = {"x": 23, "y": 24, "m": 25, "mt": 25}
    c = chr_str.lower().removeprefix("chr")
    if c.isdigit():
        return (0, int(c))
    if c in _chr_map:
        return (0, _chr_map[c])
    return (1, c)  # 非标准染色体，如 chrUn、chrGL 等放到后面

def _parse(name: str) -> Tuple[str, int, int]:
    _pat = re.compile(r"^(chr[^:]+):(\d+)_(\d+)$")
    m = _pat.match(name.strip())
    if not m:
        raise ValueError(f"非法区间格式: {name!r}，期望形如 'chr8:100200000_100300000'")
    chrn, s, e = m.group(1), int(m.group(2)), int(m.group(3))
    if e < s:
        s, e = e, s
    return chrn, s, e

def merge_hub_cprs(names: Iterable[str], insert: int = 0) -> List[str]:
    """
    合并同染色体上、相邻或相距 <= insert 的 CPR 区间。
    - 支持重叠与紧邻（距离<=insert）的级联合并。
    - 输入可为 df['names'] 或任意可迭代字符串。
    - 返回合并后的同格式列表。
    """
    if insert < 0:
        raise ValueError("insert 需为非负整数")

    # 解析与排序
    items = [_parse(x) for x in names]
    items.sort(key=lambda t: (_chrom_key(t[0]), t[1], t[2]))

    merged: List[Tuple[str, int, int]] = []
    for chrn, s, e in items:
        if not merged:
            merged.append((chrn, s, e))
            continue

        p_chrn, p_s, p_e = merged[-1]

        if chrn == p_chrn:
            # 距离定义：下一区间起点到当前区间终点的间隔，重叠时为负，紧邻为0
            gap = s - p_e
            if gap <= insert:
                # 可合并：扩展终点到更大值
                merged[-1] = (p_chrn, p_s, max(p_e, e))
                continue

        # 不能合并则追加为新段
        merged.append((chrn, s, e))

    return [f"{c}:{s}_{e}" for c, s, e in merged]

# 用法示例：
# merged_names = merge_hub_cprs(df['names'], insert=100_000)


In [56]:
### anno ecdna2gene oncogene pathway
### import ecdna2gene2pathway function

In [57]:
##### 绘制 ecDNA circle 圈图

In [58]:
#### generate circle viz pre-files
### import generate_circle_viz_pre_files function

In [59]:
echub_tb = pd.read_csv("D:/02.project/18.ecDNA/18.hezuo/ecdna.hub.cluster.xls",sep="\t")
echub_tb

,Hub,CPR,Sample,Chr,St,Cluster
0,ce321-fragments.tsv.gz:1268,chr8:140200000_140300000,ce321-fragments.tsv.gz,chr8,140200000.0,48
1,ce321-fragments.tsv.gz:1268,chr8:140300000_140400000,ce321-fragments.tsv.gz,chr8,140300000.0,48
2,ce321-fragments.tsv.gz:1268,chr15:75400000_75500000,ce321-fragments.tsv.gz,chr15,75400000.0,48
3,ce321-fragments.tsv.gz:1268,chr19:7900000_8000000,ce321-fragments.tsv.gz,chr19,7900000.0,48
4,ce321-fragments.tsv.gz:1268,chr2:57700000_57800000,ce321-fragments.tsv.gz,chr2,57700000.0,48
...,...,...,...,...,...,...
1506,vf027v1-s2-atac_fragments.tsv.gz:1774,chrX:72000000_72100000,vf027v1-s2-atac_fragments.tsv.gz,chrX,72000000.0,2
1507,vf027v1-s2-atac_fragments.tsv.gz:1774,chrX:82000000_82100000,vf027v1-s2-atac_fragments.tsv.gz,chrX,82000000.0,2
1508,vf027v1-s2-atac_fragments.tsv.gz:1774,chrX:83600000_83700000,vf027v1-s2-atac_fragments.tsv.gz,chrX,83600000.0,2
1509,vf027v1-s2-atac_fragments.tsv.gz:1774,chrX:89000000_89100000,vf027v1-s2-atac_fragments.tsv.gz,chrX,89000000.0,2


In [137]:
## MYC: chr8:1277-1278 hk
## hub 48
df48 = echub_tb[echub_tb["Cluster"]==48]
print(len(df48["CPR"].unique()))
df48

140


,Hub,CPR,Sample,Chr,St,Cluster
0,ce321-fragments.tsv.gz:1268,chr8:140200000_140300000,ce321-fragments.tsv.gz,chr8,140200000.0,48
1,ce321-fragments.tsv.gz:1268,chr8:140300000_140400000,ce321-fragments.tsv.gz,chr8,140300000.0,48
2,ce321-fragments.tsv.gz:1268,chr15:75400000_75500000,ce321-fragments.tsv.gz,chr15,75400000.0,48
3,ce321-fragments.tsv.gz:1268,chr19:7900000_8000000,ce321-fragments.tsv.gz,chr19,7900000.0,48
4,ce321-fragments.tsv.gz:1268,chr2:57700000_57800000,ce321-fragments.tsv.gz,chr2,57700000.0,48
...,...,...,...,...,...,...
1411,p5797-n1-atac_fragments.tsv.gz:1604,chr8:86800000_86900000,p5797-n1-atac_fragments.tsv.gz,chr8,86800000.0,48
1412,p5797-n1-atac_fragments.tsv.gz:1604,chr8:86900000_87000000,p5797-n1-atac_fragments.tsv.gz,chr8,86900000.0,48
1413,p5797-n1-atac_fragments.tsv.gz:1604,chr8:87200000_87300000,p5797-n1-atac_fragments.tsv.gz,chr8,87200000.0,48
1414,p5797-n1-atac_fragments.tsv.gz:1604,chr8:87300000_87400000,p5797-n1-atac_fragments.tsv.gz,chr8,87300000.0,48


In [100]:
## multipe chr
## hub 159
df159 = echub_tb[echub_tb["Cluster"]==159]
print(len(df159["CPR"].unique()))
df159

,Hub,CPR,Sample,Chr,St,Cluster
131,ce339e1-s1-atac_fragments.tsv.gz:1307,chr11:112600000_112700000,ce339e1-s1-atac_fragments.tsv.gz,chr11,112600000.0,159
132,ce339e1-s1-atac_fragments.tsv.gz:1307,chr11:118200000_118300000,ce339e1-s1-atac_fragments.tsv.gz,chr11,118200000.0,159
133,ce339e1-s1-atac_fragments.tsv.gz:1307,chr11:119400000_119500000,ce339e1-s1-atac_fragments.tsv.gz,chr11,119400000.0,159
134,ce339e1-s1-atac_fragments.tsv.gz:1307,chr11:119600000_119700000,ce339e1-s1-atac_fragments.tsv.gz,chr11,119600000.0,159
135,ce339e1-s1-atac_fragments.tsv.gz:1307,chr11:119900000_120000000,ce339e1-s1-atac_fragments.tsv.gz,chr11,119900000.0,159
...,...,...,...,...,...,...
1082,ht306p1-s1h1-atac_fragments.tsv.gz:1536,chr3:100700000_100800000,ht306p1-s1h1-atac_fragments.tsv.gz,chr3,100700000.0,159
1083,ht306p1-s1h1-atac_fragments.tsv.gz:1536,chr6:17900000_18000000,ht306p1-s1h1-atac_fragments.tsv.gz,chr6,17900000.0,159
1084,ht306p1-s1h1-atac_fragments.tsv.gz:1536,chr9:123900000_124000000,ht306p1-s1h1-atac_fragments.tsv.gz,chr9,123900000.0,159
1085,ht306p1-s1h1-atac_fragments.tsv.gz:1536,chr19:4600000_4700000,ht306p1-s1h1-atac_fragments.tsv.gz,chr19,4600000.0,159


In [138]:
#### output-path
outfile_path = "D:/02.project/18.ecDNA/02.code/v0.1.1/fig_cpr_rbg_stat_2/14.Fig1-ecDNA_hub/hub48"
#outfile_path = "D:/02.project/18.ecDNA/02.code/v0.1.1/fig_cpr_rbg_stat_2/14.Fig1-ecDNA_hub/hub159"
if(os.path.exists(outfile_path) != True): os.makedirs(outfile_path)
os.chdir(outfile_path)
print(os.getcwd())

D:\02.project\18.ecDNA\02.code\v0.1.1\fig_cpr_rbg_stat_2\14.Fig1-ecDNA_hub\hub48


In [139]:
#df_hub = echub_tb[(echub_tb["Cluster"]==159)].groupby("CPR").filter(lambda x: len(x) >= 1).drop_duplicates(subset=["CPR"])
df_hub = echub_tb[(echub_tb["Cluster"]==48)].groupby("CPR").filter(lambda x: len(x) >= 1).drop_duplicates(subset=["CPR"])
### 
df_hub["chr_raw"] = df_hub["CPR"]
df_ecdna = df_hub.copy()
df_ecdna_gene = genebody_region(df_fragments=df_ecdna,species="hg38")
df_ecdna_gene['oncogene'] = df_ecdna_gene['genebody_region_gene'].apply(custom_transform)
df_ecdna_gene

species value: hg38


,Hub,CPR,Sample,Chr,St,Cluster,chr_raw,genebody_region,genebody_region_gene,oncogene
0,ce321-fragments.tsv.gz:1268,chr8:140200000_140300000,ce321-fragments.tsv.gz,chr8,140200000.0,48,chr8:140200000_140300000,1,[TRAPPC9],[]
1,ce321-fragments.tsv.gz:1268,chr8:140300000_140400000,ce321-fragments.tsv.gz,chr8,140300000.0,48,chr8:140300000_140400000,1,[TRAPPC9],[]
2,ce321-fragments.tsv.gz:1268,chr15:75400000_75500000,ce321-fragments.tsv.gz,chr15,75400000.0,48,chr15:75400000_75500000,5,"[SIN3A, CTD-2562G15.2, CTD-2562G15.3, PTPN9, C...",[]
3,ce321-fragments.tsv.gz:1268,chr19:7900000_8000000,ce321-fragments.tsv.gz,chr19,7900000.0,48,chr19:7900000_8000000,12,"[LRRC8E, CTD-3193O13.12, MAP2K7, CTD-3193O13.1...",[]
4,ce321-fragments.tsv.gz:1268,chr2:57700000_57800000,ce321-fragments.tsv.gz,chr2,57700000.0,48,chr2:57700000_57800000,1,[ACTG1P22],[]
...,...,...,...,...,...,...,...,...,...,...
1411,p5797-n1-atac_fragments.tsv.gz:1604,chr8:86800000_86900000,p5797-n1-atac_fragments.tsv.gz,chr8,86800000.0,48,chr8:86800000_86900000,2,"[RP11-386D6.2, CNBD1]",[CNBD1]
1412,p5797-n1-atac_fragments.tsv.gz:1604,chr8:86900000_87000000,p5797-n1-atac_fragments.tsv.gz,chr8,86900000.0,48,chr8:86900000_87000000,1,[CNBD1],[CNBD1]
1413,p5797-n1-atac_fragments.tsv.gz:1604,chr8:87200000_87300000,p5797-n1-atac_fragments.tsv.gz,chr8,87200000.0,48,chr8:87200000_87300000,1,[CNBD1],[CNBD1]
1414,p5797-n1-atac_fragments.tsv.gz:1604,chr8:87300000_87400000,p5797-n1-atac_fragments.tsv.gz,chr8,87300000.0,48,chr8:87300000_87400000,1,[CNBD1],[CNBD1]


In [140]:
df_ecdna_gene["oncogene"].value_counts()

[]         116
0            9
[CNBD1]      4
[LPP]        3
[PVT1]       3
[BCL6]       1
[ETV5]       1
[NF1]        1
[FGFR1]      1
[NBN]        1
Name: oncogene, dtype: int64

In [115]:
df_ecdna_gene.to_csv('./f00-df_ecdna_gene.xls', sep='\t', index=True)

In [116]:
# 合并
# Finally, CPRs originating from neighboring genomic regions were recognized as components of the same ecDNA.
# 合并策略：合并同染色体上、相邻或相距 <= insert 的 CPR 区间；取合并后长度最长的 top3 ecDNA region作为 ecDNA hub的组成结构

In [141]:
merged_names = merge_hub_cprs(df_ecdna_gene['chr_raw'], insert=100_000)

# 解析并计算长度
def interval_length(name: str) -> int:
    chrn, rng = name.split(":")
    start, end = map(int, rng.split("_"))
    return (end - start)/100000
result_df = pd.DataFrame({
    "name": merged_names,
    "length": [interval_length(x) for x in merged_names]
})

print(result_df)

                        name  length
0     chr2:57700000_57800000     1.0
1     chr2:59600000_59700000     1.0
2   chr2:170900000_171100000     2.0
3   chr2:172000000_172100000     1.0
4   chr2:172400000_172500000     1.0
..                       ...     ...
67   chr17:66500000_66700000     2.0
68   chr17:67000000_67100000     1.0
69   chr18:23800000_23900000     1.0
70     chr19:7900000_8000000     1.0
71   chr21:42400000_42500000     1.0

[72 rows x 2 columns]


In [142]:
### 
result_df["chr_raw"] = result_df["name"]
df_ecdna = result_df.copy()
df_ecdna_gene = genebody_region(df_fragments=df_ecdna,species="hg38")
df_ecdna_gene['oncogene'] = df_ecdna_gene['genebody_region_gene'].apply(custom_transform)
df_ecdna_gene

species value: hg38


,name,length,chr_raw,genebody_region,genebody_region_gene,oncogene
0,chr2:57700000_57800000,1.0,chr2:57700000_57800000,1,[ACTG1P22],[]
1,chr2:59600000_59700000,1.0,chr2:59600000_59700000,4,"[RP11-444A22.1, AC007131.2, RNU6-508P, RNA5SP94]",[]
2,chr2:170900000_171100000,2.0,chr2:170900000_171100000,2,"[GORASP2, TLK1]",[]
3,chr2:172000000_172100000,1.0,chr2:172000000_172100000,3,"[METAP1D, DLX1, DLX2]",[]
4,chr2:172400000_172500000,1.0,chr2:172400000_172500000,5,"[RP11-227L6.1, ITGA6, AC078883.4, AC078883.3, ...",[]
...,...,...,...,...,...,...
67,chr17:66500000_66700000,2.0,chr17:66500000_66700000,4,"[PRKCA, RNU6-928P, RNA5SP445, AC006947.1]",[]
68,chr17:67000000_67100000,1.0,chr17:67000000_67100000,5,"[CACNG4, RP11-349A8.3, RP11-74H8.1, CACNG1, HELZ]",[]
69,chr18:23800000_23900000,1.0,chr18:23800000_23900000,1,[LAMA3],[]
70,chr19:7900000_8000000,1.0,chr19:7900000_8000000,12,"[LRRC8E, CTD-3193O13.12, MAP2K7, CTD-3193O13.1...",[]


In [143]:
df_ecdna_gene["oncogene"].value_counts()

[]             60
[LPP]           2
0               2
[CNBD1]         2
[ETV5]          1
[BCL6]          1
[FGFR1]         1
[NBN]           1
[MYC, PVT1]     1
[NF1]           1
Name: oncogene, dtype: int64

In [120]:
df_ecdna_gene.to_csv('./f01-df_ecdna_gene_merge_hub_cprs.xls', sep='\t', index=True)

In [144]:
#ntop = 10
ntop = 3
df_hub_cc = result_df[result_df["length"]>1].nlargest(ntop, "length")
print(df_hub_cc)
df_hub_cc['names'] = df_hub_cc['name']

                        name  length                   chr_raw
44  chr8:125600000_127200000    16.0  chr8:125600000_127200000
45  chr8:127600000_128700000    11.0  chr8:127600000_128700000
35    chr8:86200000_87000000     8.0    chr8:86200000_87000000


In [145]:
circle_graph_file(df=df_hub_cc,outfile_path=outfile_path)

D:\02.project\18.ecDNA\02.code\v0.1.1\fig_cpr_rbg_stat_2\14.Fig1-ecDNA_hub\hub48
Done: amplicon_cycles.txt
Done: amplicon_graph.txt


In [146]:
#### circle graph

In [147]:
!python D:\CycleViz\CycleViz.py  --ref hg38 --cycles_file amplicon_cycles.txt --cycle 2 -g amplicon_graph.txt --rotate_to_min --figure_size_style normal --gene_subset_file Bushman

CycleViz 0.2.0
Using resources in D:\CycleViz/resources/
Plots will use reference hg38
Converting cycles file segment boundaries to graph file segment boundaries
amplicon_BPG_converted_cycles.txt
checking adjacency
plotting structure
Reading genes
plotting genes
saving PNG
saving PDF
finished



In [148]:
### hub 48
!python D:\CycleViz\CycleViz.py  --ref hg38 --cycles_file amplicon_cycles.txt --cycle 2 -g amplicon_graph.txt --rotate_to_min --figure_size_style normal --gene_highlight_list  MYC PVT1 WWP1 CNBD1 LYN
### hub 159
#!python D:\CycleViz\CycleViz.py  --ref hg38 --cycles_file amplicon_cycles.txt --cycle 2 -g amplicon_graph.txt --rotate_to_min --figure_size_style normal --gene_highlight_list SCARB1 GAS7 RAB31 POU2F2 CIC FRMD8 MALAT1 LTBP3 THY1

CycleViz 0.2.0
Using resources in D:\CycleViz/resources/
Plots will use reference hg38
Converting cycles file segment boundaries to graph file segment boundaries
amplicon_BPG_converted_cycles.txt
checking adjacency
plotting structure
Reading genes
plotting genes
saving PNG
saving PDF
finished



In [149]:
!python D:\CycleViz\CycleViz.py  --ref hg38 --cycles_file amplicon_cycles.txt --cycle 2 -g amplicon_graph.txt --rotate_to_min --figure_size_style small --gene_subset_file Bushman

CycleViz 0.2.0
Using resources in D:\CycleViz/resources/
Plots will use reference hg38
Using figure_size_style=small
Bar width scaled up 1.5x to 1.05
Gene fontsize scaled up 2x to 14
Tick fontsize scaled up 1.5x to: 6.0
Converting cycles file segment boundaries to graph file segment boundaries
amplicon_BPG_converted_cycles.txt
checking adjacency
plotting structure
Reading genes
plotting genes
saving PNG
saving PDF
finished



In [150]:
### hub 48
!python D:\CycleViz\CycleViz.py  --ref hg38 --cycles_file amplicon_cycles.txt --cycle 2 -g amplicon_graph.txt --rotate_to_min --figure_size_style small --gene_highlight_list  MYC PVT1 WWP1 CNBD1 LYN
### hub 159
#!python D:\CycleViz\CycleViz.py  --ref hg38 --cycles_file amplicon_cycles.txt --cycle 2 -g amplicon_graph.txt --rotate_to_min --figure_size_style small --gene_highlight_list SCARB1 GAS7 RAB31 POU2F2 CIC FRMD8 MALAT1 LTBP3 THY1

CycleViz 0.2.0
Using resources in D:\CycleViz/resources/
Plots will use reference hg38
Using figure_size_style=small
Bar width scaled up 1.5x to 1.05
Gene fontsize scaled up 2x to 14
Tick fontsize scaled up 1.5x to: 6.0
Converting cycles file segment boundaries to graph file segment boundaries
amplicon_BPG_converted_cycles.txt
checking adjacency
plotting structure
Reading genes
plotting genes
saving PNG
saving PDF
finished

